# GeoAI Fine-Tuning with Unsloth 🚀
This notebook fine-tunes the `Qwen/Qwen2.5-Coder-1.5B-Instruct` model on your custom GeoAI dataset using Google Colab's free T4 GPU.
Using Unsloth makes training 2x faster and uses 70% less memory!

**Instructions:**
1. Upload your generated `geoai_train.jsonl` to the Colab files section (left sidebar).
2. Go to **Runtime -> Change runtime type** and ensure Hardware Accelerator is set to **T4 GPU**.
3. Run all cells!

In [ ]:
%%capture
!pip install unsloth
!pip install unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git
!pip install --no-deps xformers trl peft accelerate bitsandbytes

## 1. Load Model & Tokenizer
We load the 1.5B Coder model in 4-bit quantization to fit easily in the 16GB VRAM limit of a T4.

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## 2. Apply LoRA Adapters
We apply LoRA to update a small percentage of weights.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

## 3. Load & Format Dataset
We will use HuggingFace's Chat Template format as the dataset is already in chat completion style.

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5", # Map to Qwen format
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts, }

from datasets import load_dataset
# IMPORTANT: Make sure you've uploaded geoai_train.jsonl to the Colab files!
dataset = load_dataset("json", data_files="geoai_train.jsonl", split="train")

dataset = dataset.map(formatting_prompts_func, batched = True,)

## 4. Train the Model!
Setup the SFTTrainer and start fine-tuning. This should take around 10-20 minutes.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 150, # You can increase this or use num_train_epochs = 1 for a full run
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 5. Export to Ollama / GGUF
Let's export the model to GGUF format so we can run it locally with Ollama!

In [ ]:
# Save to 16bit GGUF (uncomment if you want this)
# model.save_pretrained_gguf("geoai_qwen2.5", tokenizer, quantization_method = "f16")

# Save to q4_k_m GGUF (recommended for running locally on smaller GPUs)
model.save_pretrained_gguf("geoai_qwen2.5", tokenizer, quantization_method = "q4_k_m")

print("Export complete! You can download the .gguf file from the left sidebar and load it into Ollama.")